# Setup

In [1]:
%pip install -q -U "sentence-transformers>=6.1.0" accelerate pillow chromadb
%pip install -q -U "git+https://github.com/huggingface/transformers.git"
%pip install -q -U bitsandbytes accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 12.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 73.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 73.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 11.5 MB/s eta 0:00:00

In [ ]:
import torch
import numpy as np
import json
import gc
import os
import sys
import time
import re
import random
import subprocess
import hashlib
import chromadb
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter
from sentence_transformers import SentenceTransformer
import transformers
import bitsandbytes as bnb
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig
)
from src.metrics import retrieval_metrics
from src.config import load_config


In [3]:

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Number of GPUs:", torch.cuda.device_count())
    
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No GPU detected!")

PyTorch version: 2.11.0+cu128
CUDA available: True
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [ ]:
preferred_pack = Path(
    "/kaggle/input/datasets/aminheari/anlp-course-pack/course_pack_v1"
)

if preferred_pack.is_dir():
    DATA_DIR = preferred_pack
else:
    candidates = [
        p.parent
        for p in Path("/kaggle/input").rglob("corpus.jsonl")
        if (p.parent / "questions_dev.jsonl").is_file()
        and (p.parent / "labels_dev.jsonl").is_file()
    ]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Could not uniquely identify course pack: {candidates}"
        )
    DATA_DIR = candidates[0]


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


corpus = load_jsonl(DATA_DIR / "corpus.jsonl")
questions = load_jsonl(DATA_DIR / "questions_dev.jsonl")
questions_dev = questions
labels = load_jsonl(DATA_DIR / "labels_dev.jsonl")
dev_questions = questions
dev_labels = labels

doc_ids = {str(d["doc_id"]) for d in corpus}
qids = {q["qid"] for q in questions}
label_qids = {row["qid"] for row in labels}

doc_map = {str(d["doc_id"]): d for d in corpus}
label_map = {row["qid"]: row for row in labels}
gold_label_map = label_map
oracle_doc_map = doc_map

assert len(doc_ids) == len(corpus), "Duplicate document IDs found."
assert qids == label_qids, "Dev questions and labels do not match."
assert all(
    {str(doc_id) for doc_id in row["gold_doc_ids"]}.issubset(doc_ids)
    for row in labels
), "At least one gold document ID is missing from the corpus."

print("Course pack:", DATA_DIR)
print("Documents:", len(corpus))
print("Dev questions:", len(questions))
print("Dev labels:", len(labels))
print("Question types:")
for qtype, count in Counter(q["type"] for q in questions).items():
    print(f"  {qtype}: {count}")
print("Dataset validation: PASSED")


Course pack: /kaggle/input/datasets/hesamgh/nlp-src/course_pack_v1
Documents: 3882
Dev questions: 300
Dev labels: 300
Question types:
  comparison: 73
  bridge_comparison: 66
  compositional: 125
  inference: 36
Dataset validation: PASSED


In [ ]:
def locate_project_root():
    options = []
    if os.environ.get("A1_PROJECT_ROOT"):
        options.append(Path(os.environ["A1_PROJECT_ROOT"]))
    options.extend([Path.cwd(), *Path.cwd().parents])
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        options.extend(p.parent.parent for p in kaggle_input.rglob("configs/dense.yaml"))
    for root in options:
        if (root / "src" / "config.py").is_file() and all(
            (root / "configs" / f"{name}.yaml").is_file()
            for name in ("closed_book", "oracle", "dense")
        ):
            return root.resolve()
    raise FileNotFoundError(
        "A1 project files not found. Set A1_PROJECT_ROOT or attach "
        "the repository files as a Kaggle Input dataset."
    )

PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CLOSED_CFG = load_config(PROJECT_ROOT / "configs/closed_book.yaml")
ORACLE_CFG = load_config(PROJECT_ROOT / "configs/oracle.yaml")
DENSE_CFG = load_config(PROJECT_ROOT / "configs/dense.yaml")
DENSE_K = int(DENSE_CFG["settings"]["retrieval"]["k"])
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else PROJECT_ROOT


def prediction_path(cfg, split):
    return OUTPUT_ROOT / cfg["source_predictions"][split]


def retrieval_path(split):
    return OUTPUT_ROOT / DENSE_CFG["retrieval_predictions"][split]

assert CLOSED_CFG["models"]["llm"]["name"] == ORACLE_CFG["models"]["llm"]["name"]
assert ORACLE_CFG["models"]["llm"]["name"] == DENSE_CFG["models"]["llm"]["name"]


def load_prompt(cfg):
    path = PROJECT_ROOT / cfg["settings"]["prompt_file"]
    return path.read_text(encoding="utf-8").strip()


def get_torch_dtype(torch, cfg):
    name = cfg["models"]["llm"]["compute_dtype"]
    if name not in {"float16", "bfloat16", "float32"}:
        raise ValueError(f"Unsupported torch dtype: {name}")
    return getattr(torch, name)

print("A1 loaded three YAML configurations from:", PROJECT_ROOT)

RETRIEVAL_METRIC_NAMES = (
    "recall@2", "hitall@2", "recall@5", "hitall@5",
    "recall@10", "hitall@10", "mrr", "ndcg@10",
)


def locate_score_script():
    preferred = [
        PROJECT_ROOT / "score.py",
        Path("/kaggle/input/datasets/hesamgh/nlp-codes/score.py"),
    ]
    for candidate in preferred:
        if candidate.is_file():
            return candidate.resolve()

    candidates = sorted({
        p.resolve()
        for p in Path("/kaggle/input").rglob("score.py")
        if p.is_file()
    })
    if len(candidates) == 1:
        return candidates[0]
    raise FileNotFoundError(
        "Could not uniquely identify score.py. Candidates: "
        + repr([str(p) for p in candidates])
    )


SCORE_PATH = locate_score_script()
print("Official scorer:", SCORE_PATH)

def run_score_cli(
    predictions_path,
    split="dev",
    partial=False,
    validate_only=False,
    stage_name=None,
    compare_path=None,
):
    """Run the official score.py CLI, optionally comparing two systems."""
    global SCORE_STDOUT, SCORE_PREDICTIONS_PATH, SCORE_SPLIT

    predictions_path = Path(predictions_path)

    if not predictions_path.is_file():
        raise FileNotFoundError(
            f"Predictions not found: {predictions_path}"
        )

    if split == "hidden" and not validate_only:
        raise ValueError(
            "Hidden labels are unavailable; use validate_only=True."
        )

    if partial and split != "dev":
        raise ValueError("--partial is supported only for dev.")

    if compare_path is not None:
        compare_path = Path(compare_path)

        if split != "dev":
            raise ValueError(
                "Paired comparison is allowed only on labeled Dev."
            )

        if not compare_path.is_file():
            raise FileNotFoundError(
                f"Comparison predictions not found: {compare_path}"
            )

        if partial or validate_only:
            raise ValueError(
                "Do not combine --compare with partial or validate-only."
            )

    command = [
        sys.executable,
        str(SCORE_PATH),
        str(predictions_path),
        "--pack", str(DATA_DIR),
        "--split", split,
    ]

    if partial:
        command.append("--partial")

    if validate_only:
        command.append("--validate-only")

    if compare_path is not None:
        command.extend(["--compare", str(compare_path)])

    if stage_name:
        print(f"\n=== Official score.py: {stage_name} ===")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
    )

    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")

    if result.stderr:
        print("STDERR:\n", result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"score.py failed (exit={result.returncode}) "
            f"for {predictions_path}"
        )

    SCORE_STDOUT = result.stdout
    SCORE_PREDICTIONS_PATH = predictions_path
    SCORE_SPLIT = split

    return result.stdout


def parse_official_overall(stdout):
    block_match = re.search(
        r"(?ms)^overall \(n=(\d+)\)\s*\n(.*?)(?=^type:|^paired difference:|\Z)",
        stdout,
    )
    if not block_match:
        raise ValueError("Could not find the overall metric block in score.py output.")

    count = int(block_match.group(1))
    block = block_match.group(2)
    official = {}
    metric_pattern = re.compile(
        r"^\s*(recall@2|hitall@2|recall@5|hitall@5|"
        r"recall@10|hitall@10|mrr|ndcg@10)\s+"
        r"([-+]?\d+(?:\.\d+)?)\s+\[",
    )
    for line in block.splitlines():
        match = metric_pattern.match(line)
        if match:
            official[match.group(1)] = float(match.group(2))

    missing = set(RETRIEVAL_METRIC_NAMES) - set(official)
    if missing:
        raise ValueError(
            f"Could not parse these retrieval metrics from score.py: {sorted(missing)}"
        )
    return count, official


def compare_retrieval_metrics_with_score(
    predictions_path,
    score_stdout,
    labels_by_qid,
    stage_name,
    tolerance_pp=0.051,
):

    predictions = load_jsonl(predictions_path)
    official_n, official = parse_official_overall(score_stdout)

    if len(predictions) != official_n:
        raise AssertionError(
            f"{stage_name}: predictions={len(predictions)} but score.py reports n={official_n}"
        )
    if not predictions:
        raise ValueError(f"{stage_name}: no predictions to compare")

    per_metric = {name: [] for name in RETRIEVAL_METRIC_NAMES}
    for prediction in predictions:
        qid = prediction["qid"]
        if qid not in labels_by_qid:
            raise KeyError(f"{stage_name}: no gold label for qid={qid}")
        if not isinstance(prediction.get("retrieved"), list):
            raise TypeError(f"{stage_name}: retrieved must be a list for qid={qid}")

        ours = retrieval_metrics(
            prediction["retrieved"],
            labels_by_qid[qid]["gold_doc_ids"],
        )
        for name in RETRIEVAL_METRIC_NAMES:
            per_metric[name].append(float(ours[name]))

    print(f"\n=== Manual comparison: {stage_name} (n={official_n}) ===")
    print(f"{'metric':<12} {'ours (%)':>12} {'score.py (%)':>14} {'delta (pp)':>12} {'status':>18}")
    print("-" * 72)

    mismatches = []
    report = []
    for name in RETRIEVAL_METRIC_NAMES:
        ours_pct = 100.0 * sum(per_metric[name]) / len(per_metric[name])
        official_pct = official[name]
        delta = ours_pct - official_pct
        status = "MATCH (rounding)" if abs(delta) <= tolerance_pp else "MISMATCH"
        if status == "MISMATCH":
            mismatches.append((name, delta))
        report.append({
            "metric": name,
            "ours_percent": ours_pct,
            "score_py_percent": official_pct,
            "delta_percentage_points": delta,
            "status": status,
        })
        print(f"{name:<12} {ours_pct:12.4f} {official_pct:14.1f} {delta:12.4f} {status:>18}")

    if mismatches:
        raise AssertionError(
            f"{stage_name}: retrieval metric mismatch against score.py: {mismatches}"
        )
    print("Result: all eight retrieval metrics match score.py within output rounding.")
    return report

def read_jsonl(path):
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(f"Predictions file not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

A1 loaded three YAML configurations from: /kaggle/input/datasets/hesamgh/nlp-src
Official scorer: /kaggle/input/datasets/hesamgh/nlp-src/score.py


# Shared LLM Setup

The configured LLM is reused for the closed-book and Oracle stages.


In [ ]:
MODEL_ID = ORACLE_CFG["models"]["llm"]["name"]

quant_config = BitsAndBytesConfig(
    load_in_4bit=ORACLE_CFG["models"]["llm"]["load_in_4bit"],
    bnb_4bit_quant_type=ORACLE_CFG["models"]["llm"]["quantization"],
    bnb_4bit_use_double_quant=ORACLE_CFG["models"]["llm"]["double_quantization"],
    bnb_4bit_compute_dtype=get_torch_dtype(torch, ORACLE_CFG)
)

start = time.perf_counter()

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading Qwen3.5-9B in 4-bit...")

llm = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=getattr(torch, ORACLE_CFG["models"]["llm"]["model_dtype"]),
    device_map=ORACLE_CFG["models"]["llm"]["device_map"],
    attn_implementation=ORACLE_CFG["models"]["llm"]["attention_implementation"],
    low_cpu_mem_usage=ORACLE_CFG["models"]["llm"]["low_cpu_mem_usage"]
)

llm.eval()

print("\nModel loaded successfully!")
print("Loading time:", round(time.perf_counter() - start, 1), "seconds")
print("Device map:", llm.hf_device_map)

for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    print(f"GPU {i} allocated: {allocated:.2f} GB")


Loading processor...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/79.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]


Model loaded successfully!
Loading time: 159.3 seconds
Device map: {'model.visual': 0, 'model.language_model.embed_tokens': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 0, 'model.language_model.layers.8': 0, 'model.language_model.layers.9': 0, 'model.language_model.layers.10': 0, 'model.language_model.layers.11': 0, 'model.language_model.layers.12': 0, 'model.language_model.layers.13': 1, 'model.language_model.layers.14': 1, 'model.language_model.layers.15': 1, 'model.language_model.layers.16': 1, 'model.language_model.layers.17': 1, 'model.language_model.layers.18': 1, 'model.language_model.layers.19': 1, 'model.language_model.layers.20': 1, 'model.language_model.layers.21': 1, 'model.language_model.layers.22': 1, 'model.language_model.layers.23

In [ ]:
ORACLE_SYSTEM_PROMPT = load_prompt(ORACLE_CFG)
MAX_INPUT_TOKENS = ORACLE_CFG["settings"]["generation"]["max_input_tokens"]

def build_oracle_messages(question):
    qid = question["qid"]
    gold_ids = gold_label_map[qid]["gold_doc_ids"]
    contexts = []

    for rank, doc_id in enumerate(gold_ids, start=1):
        doc = oracle_doc_map[str(doc_id)]
        contexts.append(
            f"[Passage {rank} | {doc_id}]\n"
            f"Title: {doc['title']}\n"
            f"Text: {doc['text']}"
        )

    user_prompt = (
        "Passages:\n\n" + "\n\n".join(contexts)
        + "\n\nQuestion: " + question["question"]
        + "\n\nFinal answer:"
    )
    return [
        {"role": "system", "content": ORACLE_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


def tokenize_oracle_prompt(question):
    return processor.apply_chat_template(
        build_oracle_messages(question),
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=ORACLE_CFG["settings"]["generation"]["enable_thinking"],
        return_dict=True,
        return_tensors="pt",
    )


sample = questions[0]
sample_messages = build_oracle_messages(sample)
print("Oracle prompt example:", sample["qid"])
print("Gold documents:", label_map[sample["qid"]]["gold_doc_ids"])
print("\nPrompt preview:\n")
print(sample_messages[1]["content"][:1500])


Oracle prompt example: q0001
Gold documents: ['p02917', 'p03365']

Prompt preview:

Passages:

[Passage 1 | p02917]
Title: Ride a Crooked Trail
Text: Ride a Crooked Trail is a 1958 American Eastmancolor Western film shot in CinemaScope, with former World War II hero Audie Murphy and future Academy Award- winning actor Walter Matthau heading a strong if not well- known cast. Gia Scala and Henry Silva co-star in the film, directed by Jesse Hibbs.

[Passage 2 | p03365]
Title: The Glassworker
Text: The Glassworker is an upcoming Pakistani Urdu- language animated film directed by Usman Riaz. It is Pakistan's first hand drawn anime film. The teaser of the film was released in February 2016. The trailer of the film was released in October 2016 and the length of the film is expected to be over an hour.

Question: Did the movies Ride A Crooked Trail and The Glassworker, originate from the same country?

Final answer:


In [ ]:
@torch.inference_mode()
def generate_oracle_answer(question, max_new_tokens=None):
    if max_new_tokens is None:
        max_new_tokens = ORACLE_CFG["settings"]["generation"]["max_new_tokens"]

    inputs = tokenize_oracle_prompt(question)
    input_length = inputs["input_ids"].shape[-1]
    if input_length > MAX_INPUT_TOKENS:
        raise ValueError(
            f"{question['qid']} exceeds context limit: "
            f"{input_length} > {MAX_INPUT_TOKENS}"
        )

    inputs = inputs.to(ORACLE_CFG["settings"]["generation"]["device"])
    outputs = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=ORACLE_CFG["settings"]["generation"]["do_sample"],
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    generated_ids = outputs[0][input_length:]
    raw_answer = processor.tokenizer.decode(
        generated_ids, skip_special_tokens=True
    ).strip()
    
    answer = re.sub(
        r"^(final answer|answer)\s*:\s*", "", raw_answer,
        flags=re.IGNORECASE,
    ).strip()
    lines = [line.strip() for line in answer.splitlines() if line.strip()]
    return lines[0].strip('"') if lines else ""


In [ ]:
EXPERIMENT_DIR = Path("/kaggle/working/llm_experiments/oracle_50")
QWEN_DIR = EXPERIMENT_DIR / "qwen3_5_9b"
QWEN_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PATH = EXPERIMENT_DIR / "sample_qids.json"
PRED_PATH = QWEN_DIR / "predictions.jsonl"
TIME_PATH = QWEN_DIR / "timings.jsonl"

SAMPLE_COUNTS = {
    "comparison": 12,
    "bridge_comparison": 11,
    "compositional": 21,
    "inference": 6,
}

rng = random.Random(42)
selected_qids = []
for qtype, n in SAMPLE_COUNTS.items():
    candidates = sorted(
        q["qid"] for q in questions if q["type"] == qtype
    )
    if len(candidates) < n:
        raise ValueError(f"Not enough dev questions for type={qtype}: {len(candidates)} < {n}")
    selected_qids.extend(rng.sample(candidates, n))

question_map = {q["qid"]: q for q in questions}
assert len(selected_qids) == 50
assert len(set(selected_qids)) == 50
assert all(qid in question_map for qid in selected_qids)
SAMPLE_PATH.write_text(json.dumps(selected_qids, indent=2), encoding="utf-8")

print("Fresh Oracle sample selected:", len(selected_qids))
print("Distribution:", dict(Counter(question_map[qid]["type"] for qid in selected_qids)))
print("Sample manifest overwritten:", SAMPLE_PATH)


Fresh Oracle sample selected: 50
Distribution: {'comparison': 12, 'bridge_comparison': 11, 'compositional': 21, 'inference': 6}
Sample manifest overwritten: /kaggle/working/llm_experiments/oracle_50/sample_qids.json


In [ ]:
PRED_PATH.unlink(missing_ok=True)
TIME_PATH.unlink(missing_ok=True)

completed = 0
with open(PRED_PATH, "w", encoding="utf-8") as pred_file, \
     open(TIME_PATH, "w", encoding="utf-8") as time_file:
    for index, qid in enumerate(selected_qids, start=1):
        q = question_map[qid]
        print(f"\n[{index}/50] {qid} | {q['type']}")
        start = time.perf_counter()
        answer = generate_oracle_answer(q)
        elapsed = time.perf_counter() - start

        prediction = {
            "qid": qid,
            "answer": answer,
            "retrieved": label_map[qid]["gold_doc_ids"],
        }
        timing = {"qid": qid, "elapsed_seconds": round(elapsed, 3)}
        pred_file.write(json.dumps(prediction, ensure_ascii=False) + "\n")
        pred_file.flush()
        time_file.write(json.dumps(timing, ensure_ascii=False) + "\n")
        time_file.flush()
        completed += 1
        print("Predicted:", answer)
        print("Gold:", label_map[qid]["answer"])
        print("Time:", round(elapsed, 2), "seconds")
        print("Freshly generated:", completed, "/ 50")

print("\nOracle 50-question generation completed:", completed, "/ 50")
print("Predictions:", PRED_PATH)



[1/50] q0077 | comparison


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Predicted: Yes
Gold: yes
Time: 6.78 seconds
Freshly generated: 1 / 50

[2/50] q0009 | comparison
Predicted: Junior Mandrake
Gold: Junior Mandrake
Time: 1.58 seconds
Freshly generated: 2 / 50

[3/50] q0171 | comparison
Predicted: Yes
Gold: yes
Time: 1.25 seconds
Freshly generated: 3 / 50

[4/50] q0156 | comparison
Predicted: Femme Like U (Donne-Moi Ton Corps)
Gold: Femme Like U (Donne-Moi Ton Corps)
Time: 2.59 seconds
Freshly generated: 4 / 50

[5/50] q0148 | comparison
Predicted: Defeat of the Mafia
Gold: Defeat Of The Mafia
Time: 1.93 seconds
Freshly generated: 5 / 50

[6/50] q0082 | comparison
Predicted: Glass Trap
Gold: Glass Trap
Time: 1.32 seconds
Freshly generated: 6 / 50

[7/50] q0075 | comparison
Predicted: Yes
Gold: yes
Time: 1.37 seconds
Freshly generated: 7 / 50

[8/50] q0062 | comparison
Predicted: Los Amantes del Señor de la Noche
Gold: Los Amantes Del Señor De La Noche
Time: 2.2 seconds
Freshly generated: 8 / 50

[9/50] q0241 | comparison
Predicted: País do Desejo
Gold: P

In [ ]:
saved_predictions = read_jsonl(PRED_PATH)
assert len(saved_predictions) == 50, (
    f"Expected 50 predictions, got {len(saved_predictions)}"
)
SCORE_STDOUT = run_score_cli(
    PRED_PATH,
    split="dev",
    partial=True,
    stage_name="Oracle 50-question sample",
)


=== Official score.py: Oracle 50-question sample ===
valid: /kaggle/working/llm_experiments/oracle_50/qwen3_5_9b/predictions.jsonl (50 predictions -- PARTIAL, not a full split)

overall (n=50)
  recall@2          89.0   [ 82.0,  94.0]
  hitall@2          78.0   [ 64.0,  88.0]
  recall@5         100.0   [100.0, 100.0]
  hitall@5         100.0   [100.0, 100.0]
  recall@10        100.0   [100.0, 100.0]
  hitall@10        100.0   [100.0, 100.0]
  mrr              100.0   [100.0, 100.0]
  ndcg@10          100.0   [100.0, 100.0]
  em                78.0   [ 66.0,  90.0]
  f1                84.0   [ 74.1,  92.5]

type: bridge_comparison (n=11)
  recall@2          50.0   [ 50.0,  50.0]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5         100.0   [100.0, 100.0]
  hitall@5         100.0   [100.0, 100.0]
  recall@10        100.0   [100.0, 100.0]
  hitall@10        100.0   [100.0, 100.0]
  mrr              100.0   [100.0, 100.0]
  ndcg@10          100.0   [100.0, 100.0]
  em              

In [ ]:
comparison_report = compare_retrieval_metrics_with_score(
    predictions_path=PRED_PATH,
    score_stdout=SCORE_STDOUT,
    labels_by_qid=label_map,
    stage_name='Oracle 50-question sample',
)


=== Manual comparison: Oracle 50-question sample (n=50) ===
metric           ours (%)   score.py (%)   delta (pp)             status
------------------------------------------------------------------------
recall@2          89.0000           89.0       0.0000   MATCH (rounding)
hitall@2          78.0000           78.0       0.0000   MATCH (rounding)
recall@5         100.0000          100.0       0.0000   MATCH (rounding)
hitall@5         100.0000          100.0       0.0000   MATCH (rounding)
recall@10        100.0000          100.0       0.0000   MATCH (rounding)
hitall@10        100.0000          100.0       0.0000   MATCH (rounding)
mrr              100.0000          100.0       0.0000   MATCH (rounding)
ndcg@10          100.0000          100.0       0.0000   MATCH (rounding)
Result: all eight retrieval metrics match score.py within output rounding.


##  Close Book System‍‍

In [ ]:
CLOSED_BOOK_SYSTEM_PROMPT = load_prompt(CLOSED_CFG)

def build_closed_book_messages(question):
    return [
        {
            "role": "system",
            "content": CLOSED_BOOK_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": question["question"]
        }
    ]

@torch.inference_mode()
def generate_closed_book_answer(question, max_new_tokens=None):
    if max_new_tokens is None:
        max_new_tokens = CLOSED_CFG["settings"]["generation"]["max_new_tokens"]

    messages = build_closed_book_messages(question)

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=CLOSED_CFG["settings"]["generation"]["enable_thinking"],
        return_dict=True,
        return_tensors="pt"
    )

    input_length = inputs["input_ids"].shape[1]

    inputs = inputs.to(CLOSED_CFG["settings"]["generation"]["device"])

    outputs = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=CLOSED_CFG["settings"]["generation"]["do_sample"],
        pad_token_id=processor.tokenizer.eos_token_id
    )

    generated_ids = outputs[0][input_length:]

    raw_answer = processor.tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    answer = re.sub(
        r"^(final answer|answer)\s*:\s*",
        "",
        raw_answer,
        flags=re.IGNORECASE
    ).strip()

    lines = [
        line.strip()
        for line in answer.splitlines()
        if line.strip()
    ]

    return lines[0].strip('"') if lines else ""

In [ ]:
RUN_DIR = prediction_path(CLOSED_CFG, "dev").parent
RUN_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = RUN_DIR / "predictions.jsonl"
TIME_PATH = RUN_DIR / "timings.jsonl"

PRED_PATH.unlink(missing_ok=True)
TIME_PATH.unlink(missing_ok=True)

with open(PRED_PATH, "w", encoding="utf-8") as pred_file, \
     open(TIME_PATH, "w", encoding="utf-8") as time_file:
    for index, q in enumerate(questions, start=1):
        start = time.perf_counter()
        answer = generate_closed_book_answer(q)
        elapsed = time.perf_counter() - start
        prediction = {"qid": q["qid"], "answer": answer, "retrieved": []}
        pred_file.write(json.dumps(prediction, ensure_ascii=False) + "\n")
        pred_file.flush()
        time_file.write(json.dumps({
            "qid": q["qid"], "elapsed_seconds": round(elapsed, 3)
        }, ensure_ascii=False) + "\n")
        time_file.flush()
        print(f"[{index}/{len(questions)}] {q['qid']} | {answer} | {elapsed:.2f}s")

fresh_rows = load_jsonl(PRED_PATH)
assert len(fresh_rows) == len(questions) == 300
assert {row["qid"] for row in fresh_rows} == {q["qid"] for q in questions}
print("Fresh Closed-book / dev predictions:", len(fresh_rows))
print("Saved to:", PRED_PATH)


[1/300] q0001 | No | 1.23s
[2/300] q0002 | Escape From L.A. | 1.59s
[3/300] q0003 | England | 1.07s
[4/300] q0004 | Henry Strutt, 2nd Baron Belper | 2.20s
[5/300] q0005 | Heart attack | 1.22s
[6/300] q0006 | Greek | 0.62s
[7/300] q0007 | Copenhagen | 1.23s
[8/300] q0008 | No | 1.13s
[9/300] q0009 | Junior Mandrake | 0.85s
[10/300] q0010 | Kentucky | 0.73s
[11/300] q0011 | Gang War In Naples | 1.61s
[12/300] q0012 | School On Fire | 0.85s
[13/300] q0013 | New York City | 0.84s
[14/300] q0014 | Paris | 1.08s
[15/300] q0015 | No | 1.15s
[16/300] q0016 | American | 0.64s
[17/300] q0017 | New York City | 0.79s
[18/300] q0018 | American | 0.62s
[19/300] q0019 | Alexander I of Russia | 0.91s
[20/300] q0020 | Lord Harcourt | 0.88s
[21/300] q0021 | Seleucus I Nicator | 1.04s
[22/300] q0022 | New York City | 1.33s
[23/300] q0023 | Snapshot Of A Crime | 0.95s
[24/300] q0024 | London | 0.62s
[25/300] q0025 | John II of France | 0.89s
[26/300] q0026 | Zürich | 1.29s
[27/300] q0027 | New York City |

In [ ]:
SCORE_STDOUT = run_score_cli(
    PRED_PATH,
    split="dev",
    stage_name="Closed-book / dev",
)


=== Official score.py: Closed-book / dev ===
valid: /kaggle/working/runs/dev/closed_book/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2           0.0   [  0.0,   0.0]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5           0.0   [  0.0,   0.0]
  hitall@5           0.0   [  0.0,   0.0]
  recall@10          0.0   [  0.0,   0.0]
  hitall@10          0.0   [  0.0,   0.0]
  mrr                0.0   [  0.0,   0.0]
  ndcg@10            0.0   [  0.0,   0.0]
  em                24.7   [ 20.0,  29.7]
  f1                29.6   [ 24.7,  35.0]

type: bridge_comparison (n=66)
  recall@2           0.0   [  0.0,   0.0]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5           0.0   [  0.0,   0.0]
  hitall@5           0.0   [  0.0,   0.0]
  recall@10          0.0   [  0.0,   0.0]
  hitall@10          0.0   [  0.0,   0.0]
  mrr                0.0   [  0.0,   0.0]
  ndcg@10            0.0   [  0.0,   0.0]
  em                40.9   [ 28.8,  53.0]
  f1                41.4   

In [ ]:
comparison_report = compare_retrieval_metrics_with_score(
    predictions_path=PRED_PATH,
    score_stdout=SCORE_STDOUT,
    labels_by_qid=label_map,
    stage_name='Closed-book / dev',
)


=== Manual comparison: Closed-book / dev (n=300) ===
metric           ours (%)   score.py (%)   delta (pp)             status
------------------------------------------------------------------------
recall@2           0.0000            0.0       0.0000   MATCH (rounding)
hitall@2           0.0000            0.0       0.0000   MATCH (rounding)
recall@5           0.0000            0.0       0.0000   MATCH (rounding)
hitall@5           0.0000            0.0       0.0000   MATCH (rounding)
recall@10          0.0000            0.0       0.0000   MATCH (rounding)
hitall@10          0.0000            0.0       0.0000   MATCH (rounding)
mrr                0.0000            0.0       0.0000   MATCH (rounding)
ndcg@10            0.0000            0.0       0.0000   MATCH (rounding)
Result: all eight retrieval metrics match score.py within output rounding.


In [ ]:
hidden_questions = load_jsonl(DATA_DIR / "questions_hidden.jsonl")
HIDDEN_DIR = prediction_path(CLOSED_CFG, "hidden").parent
HIDDEN_DIR.mkdir(parents=True, exist_ok=True)
HIDDEN_PRED = HIDDEN_DIR / "predictions.jsonl"
HIDDEN_TIME = HIDDEN_DIR / "timings.jsonl"

HIDDEN_PRED.unlink(missing_ok=True)
HIDDEN_TIME.unlink(missing_ok=True)

with open(HIDDEN_PRED, "w", encoding="utf-8") as pred_file, \
     open(HIDDEN_TIME, "w", encoding="utf-8") as time_file:
    for i, q in enumerate(hidden_questions, start=1):
        start = time.perf_counter()
        answer = generate_closed_book_answer(q)
        elapsed = time.perf_counter() - start
        prediction = {"qid": q["qid"], "answer": answer, "retrieved": []}
        pred_file.write(json.dumps(prediction, ensure_ascii=False) + "\n")
        pred_file.flush()
        time_file.write(json.dumps({
            "qid": q["qid"], "elapsed_seconds": round(elapsed, 3)
        }, ensure_ascii=False) + "\n")
        time_file.flush()
        print(f"[{i}/{len(hidden_questions)}] {q['qid']} | {answer} | {elapsed:.1f}s")

hidden_rows = load_jsonl(HIDDEN_PRED)
assert len(hidden_rows) == len(hidden_questions) == 300
assert {row["qid"] for row in hidden_rows} == {q["qid"] for q in hidden_questions}
print("Fresh Closed-book / hidden predictions:", len(hidden_rows))
print("Saved:", HIDDEN_PRED)


[1/300] q1001 | Cross Of Love | 1.0s
[2/300] q1002 | Salad By The Roots | 1.1s
[3/300] q1003 | New York City | 0.8s
[4/300] q1004 | Ek Haseena Thi Ek Deewana Tha | 1.5s
[5/300] q1005 | The Slaughter Rule | 0.9s
[6/300] q1006 | New York City | 0.8s
[7/300] q1007 | Cloud Dancer | 0.8s
[8/300] q1008 | Bucharest | 0.7s
[9/300] q1009 | Eliza Ritchie | 0.9s
[10/300] q1010 | Linda Hamilton | 0.8s
[11/300] q1011 | New Zealand | 0.7s
[12/300] q1012 | Lương Hoàng Nam | 0.9s
[13/300] q1013 | No | 0.6s
[14/300] q1014 | Nesmrtelná Teta | 1.3s
[15/300] q1015 | Italian | 0.6s
[16/300] q1016 | New York City | 0.8s
[17/300] q1017 | Alexander III | 0.7s
[18/300] q1018 | Marius Mapou | 0.9s
[19/300] q1019 | Stanford University | 0.8s
[20/300] q1020 | Anne of Austria | 0.8s
[21/300] q1021 | Charles Howard, 1st Viscount Morpeth's paternal grandfather was Charles Howard, 1st Earl of Carlisle. | 3.1s
[22/300] q1022 | Jan Zamoyski | 1.0s
[23/300] q1023 | New York City | 0.9s
[24/300] q1024 | Det Drønner Gjenn

In [ ]:
SCORE_STDOUT = run_score_cli(
    HIDDEN_PRED,
    split="hidden",
    validate_only=True,
    stage_name="Closed-book / hidden validation",
)


=== Official score.py: Closed-book / hidden validation ===
valid: /kaggle/working/runs/hidden/closed_book/predictions.jsonl (300 predictions)


### Closed-book / hidden: metric comparison intentionally skipped

`score.py --validate-only` checks the prediction schema and IDs. We do not calculate or compare retrieval metrics for Hidden because its gold labels are not available in this notebook.


# Oracle Context

In [ ]:
assert "llm" in globals() and "processor" in globals(), (
    "The configured LLM/processor is not loaded. Run the shared LLM setup cell first."
)

dev_questions = questions
dev_labels = labels
gold_label_map = label_map
oracle_doc_map = doc_map

assert len(dev_questions) == 300
assert len(dev_labels) == 300
assert set(gold_label_map) == {q["qid"] for q in dev_questions}
assert all(
    str(doc_id) in oracle_doc_map
    for row in dev_labels
    for doc_id in row["gold_doc_ids"]
)

print("Model type:", llm.config.model_type)
print("Questions:", len(dev_questions))
print("Documents:", len(corpus))
print("Labels:", len(dev_labels))
print("Dataset ready.")


Model type: qwen3_5
Questions: 300
Documents: 3882
Labels: 300
Dataset ready.


In [22]:
token_lengths = []

for i, q in enumerate(dev_questions, start=1):
    inputs = tokenize_oracle_prompt(q)
    n_tokens = inputs["input_ids"].shape[-1]
    token_lengths.append((q["qid"], n_tokens))

    if i % 50 == 0:
        print(f"Checked {i}/300 Oracle prompts")

too_long = [
    (qid, n_tokens)
    for qid, n_tokens in token_lengths
    if n_tokens > MAX_INPUT_TOKENS
]

print("Maximum input tokens:", max(n for _, n in token_lengths))
print("Prompts over limit:", len(too_long))
if too_long:
    print("First long prompts:", too_long[:20])
    raise ValueError(
        "Some Oracle prompts exceed the configured input limit. "
        "Review the context before starting generation."
    )
print("All Oracle prompts fit within the configured limit.")


Checked 50/300 Oracle prompts
Checked 100/300 Oracle prompts
Checked 150/300 Oracle prompts
Checked 200/300 Oracle prompts
Checked 250/300 Oracle prompts
Checked 300/300 Oracle prompts
Maximum input tokens: 1390
Prompts over limit: 0
All Oracle prompts fit within the configured limit.


In [ ]:
RUN_DIR = prediction_path(ORACLE_CFG, "dev").parent
RUN_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = RUN_DIR / "predictions.jsonl"
TIME_PATH = RUN_DIR / "timings.jsonl"

PRED_PATH.unlink(missing_ok=True)
TIME_PATH.unlink(missing_ok=True)
expected_qids = {q["qid"] for q in dev_questions}

with open(PRED_PATH, "w", encoding="utf-8") as pred_file, \
     open(TIME_PATH, "w", encoding="utf-8") as time_file:
    for index, q in enumerate(dev_questions, start=1):
        start = time.perf_counter()
        answer = generate_oracle_answer(q)
        elapsed = time.perf_counter() - start
        prediction = {
            "qid": q["qid"],
            "answer": answer,
            "retrieved": gold_label_map[q["qid"]]["gold_doc_ids"],
        }
        timing = {"qid": q["qid"], "elapsed_seconds": round(elapsed, 3)}
        pred_file.write(json.dumps(prediction, ensure_ascii=False) + "\n")
        pred_file.flush()
        time_file.write(json.dumps(timing, ensure_ascii=False) + "\n")
        time_file.flush()
        print(f"[{index}/300] {q['qid']} | {elapsed:.1f}s | {answer[:100]}")

fresh_oracle = load_jsonl(PRED_PATH)
assert len(fresh_oracle) == 300
assert {row["qid"] for row in fresh_oracle} == expected_qids
print("Fresh Oracle / dev completed:", len(fresh_oracle))
print("Predictions:", PRED_PATH)

[1/300] q0001 | 1.3s | No
[2/300] q0002 | 2.3s | The Third Side Of The River
[3/300] q0003 | 1.0s | England
[4/300] q0004 | 1.4s | Nina Škottová
[5/300] q0005 | 2.7s | He died from complications of Huntington's disease.
[6/300] q0006 | 1.7s | Greek-Swedish
[7/300] q0007 | 1.4s | Århus
[8/300] q0008 | 1.3s | yes
[9/300] q0009 | 1.0s | Junior Mandrake
[10/300] q0010 | 2.5s | Georgetown, Kentucky
[11/300] q0011 | 1.3s | Gang War In Naples
[12/300] q0012 | 2.1s | School On Fire
[13/300] q0013 | 1.6s | New York City
[14/300] q0014 | 1.7s | Barcelona, Spain
[15/300] q0015 | 1.7s | No
[16/300] q0016 | 1.1s | Austrian
[17/300] q0017 | 1.3s | Florence
[18/300] q0018 | 1.3s | Italian-American
[19/300] q0019 | 2.3s | Joseph, Duke of Saxe-Altenburg
[20/300] q0020 | 1.7s | Henry Vernon
[21/300] q0021 | 1.2s | Antiochus II
[22/300] q0022 | 1.8s | Johannesburg, South Africa
[23/300] q0023 | 1.2s | Snapshot of a Crime
[24/300] q0024 | 1.3s | Weston-super-Mare
[25/300] q0025 | 2.1s | John I, Duke of Bo

In [ ]:
saved_oracle_predictions = read_jsonl(PRED_PATH)
assert len(saved_oracle_predictions) == 300

SCORE_STDOUT = run_score_cli(
    PRED_PATH,
    split="dev",
    stage_name="Oracle context / dev",
)


=== Official score.py: Oracle context / dev ===
valid: /kaggle/working/runs/dev/context_oracle_qwen3_5_9b/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2          89.0   [ 86.5,  91.3]
  hitall@2          78.0   [ 73.0,  82.7]
  recall@5         100.0   [100.0, 100.0]
  hitall@5         100.0   [100.0, 100.0]
  recall@10        100.0   [100.0, 100.0]
  hitall@10        100.0   [100.0, 100.0]
  mrr              100.0   [100.0, 100.0]
  ndcg@10          100.0   [100.0, 100.0]
  em                70.3   [ 65.0,  75.7]
  f1                77.0   [ 72.4,  81.3]

type: bridge_comparison (n=66)
  recall@2          50.0   [ 50.0,  50.0]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5         100.0   [100.0, 100.0]
  hitall@5         100.0   [100.0, 100.0]
  recall@10        100.0   [100.0, 100.0]
  hitall@10        100.0   [100.0, 100.0]
  mrr              100.0   [100.0, 100.0]
  ndcg@10          100.0   [100.0, 100.0]
  em                72.7   [ 62.1,  81.8]
  f1      

In [ ]:
comparison_report = compare_retrieval_metrics_with_score(
    predictions_path=PRED_PATH,
    score_stdout=SCORE_STDOUT,
    labels_by_qid=label_map,
    stage_name='Oracle context / dev',
)


=== Manual comparison: Oracle context / dev (n=300) ===
metric           ours (%)   score.py (%)   delta (pp)             status
------------------------------------------------------------------------
recall@2          89.0000           89.0       0.0000   MATCH (rounding)
hitall@2          78.0000           78.0       0.0000   MATCH (rounding)
recall@5         100.0000          100.0       0.0000   MATCH (rounding)
hitall@5         100.0000          100.0       0.0000   MATCH (rounding)
recall@10        100.0000          100.0       0.0000   MATCH (rounding)
hitall@10        100.0000          100.0       0.0000   MATCH (rounding)
mrr              100.0000          100.0       0.0000   MATCH (rounding)
ndcg@10          100.0000          100.0       0.0000   MATCH (rounding)
Result: all eight retrieval metrics match score.py within output rounding.


# Dense Retrieval

In [ ]:
questions_dev = questions

doc_ids_original = [str(d["doc_id"]) for d in corpus]
assert len(corpus) == 3882, f"Unexpected corpus size: {len(corpus)}"
assert len(set(doc_ids_original)) == len(doc_ids_original)
assert len(questions_dev) == 300

document_texts = [
    f"Title: {d['title']}\nText: {d['text']}"
    for d in corpus
]

hasher = hashlib.sha256()
for doc in corpus:
    for field in ("doc_id", "title", "text"):
        hasher.update(str(doc[field]).encode("utf-8"))
        hasher.update(b"\0")
CORPUS_SHA256 = hasher.hexdigest()

print("Course pack:", DATA_DIR)
print("Corpus passages:", len(corpus))
print("Dev questions:", len(questions_dev))
print("Corpus fingerprint:", CORPUS_SHA256[:16])
print("ChromaDB version:", chromadb.__version__)


Course pack: /kaggle/input/datasets/hesamgh/nlp-src/course_pack_v1
Corpus passages: 3882
Dev questions: 300
Corpus fingerprint: 5c8485290f4bf61f
ChromaDB version: 1.5.9


In [ ]:
if "llm" in globals():
    del llm

if "model" in globals() and type(model).__name__ == "SentenceTransformer":
    del model

gc.collect()
torch.cuda.empty_cache()

In [28]:
EMBED_MODEL_ID = DENSE_CFG["models"]["embedder"]

embedding_model = SentenceTransformer(
    EMBED_MODEL_ID,
    model_kwargs={"torch_dtype": getattr(torch, DENSE_CFG["settings"]["embedding"]["torch_dtype"])},
    device=DENSE_CFG["settings"]["embedding"]["device"]
)

EMBED_DIM = embedding_model.get_sentence_embedding_dimension()

print("Embedding model:", EMBED_MODEL_ID)
print("Embedding dimension:", EMBED_DIM)
print("Max sequence length:", embedding_model.max_seq_length)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Embedding model: Qwen/Qwen3-Embedding-4B
Embedding dimension: 2560
Max sequence length: 40960


/tmp/ipykernel_50/2897231317.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM = embedding_model.get_sentence_embedding_dimension()


In [ ]:
CHROMA_DIR = Path(
    DENSE_CFG["settings"]["retrieval"]["chroma_dir"]
)

COLLECTION_NAME = DENSE_CFG["settings"]["retrieval"]["collection_name"]

client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

COLLECTION_METADATA = {
    "hnsw:space": DENSE_CFG["settings"]["retrieval"]["space"],
    "embedding_model": EMBED_MODEL_ID,
    "corpus_sha256": CORPUS_SHA256,
    "embedding_dim": int(EMBED_DIM),
    "l2_normalized": True
}


def collection_is_valid(col):
    metadata = col.metadata or {}

    return (
        all(
            metadata.get(key) == value
            for key, value in COLLECTION_METADATA.items()
        )
        and col.count() == len(corpus)
    )


collection = None

try:
    existing = client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=None
    )

    if collection_is_valid(existing):
        collection = existing
        print("Reusing existing persistent collection.")
    else:
        print("Existing collection is stale; rebuilding.")
        client.delete_collection(name=COLLECTION_NAME)

except Exception:
    print("No reusable collection found.")

if collection is None:
    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata=COLLECTION_METADATA,
        embedding_function=None
    )

    document_embeddings = embedding_model.encode(
        document_texts,
        batch_size=DENSE_CFG["settings"]["embedding"]["document_batch_size"],
        precision=DENSE_CFG["settings"]["embedding"]["precision"],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=DENSE_CFG["settings"]["retrieval"]["normalize_embeddings"]
    )

    document_embeddings

    CHROMA_BATCH_SIZE = DENSE_CFG["settings"]["retrieval"]["chroma_batch_size"]

    for start in range(0, len(corpus), CHROMA_BATCH_SIZE):
        end = min(start + CHROMA_BATCH_SIZE, len(corpus))

        batch_docs = corpus[start:end]

        collection.upsert(
            ids=doc_ids_original[start:end],
            embeddings=document_embeddings[start:end].tolist(),
            documents=document_texts[start:end],
            metadatas=[
                {
                    "title": str(d["title"]),
                    "doc_id": str(d["doc_id"])
                }
                for d in batch_docs
            ]
        )

        print(
            f"Stored {end}/{len(corpus)} passages",
            end="\r"
        )

    del document_embeddings
    gc.collect()

assert collection.count() == len(corpus)

print("\nCollection ready.")
print("Stored passages:", collection.count())
print("Persistent directory:", CHROMA_DIR)

No reusable collection found.


Batches:   0%|          | 0/971 [00:00<?, ?it/s]

Stored 3882/3882 passages
Collection ready.
Stored passages: 3882
Persistent directory: /kaggle/working/chroma_qwen3_embedding4b


In [ ]:
QUERY_INSTRUCTION = DENSE_CFG["settings"]["embedding"]["query_instruction"]

def embed_query(query:str):
    
    formatted_query = (
        f"Instruct: {QUERY_INSTRUCTION}\n"
        f"Query:{query}"
    )
    
    embed_query = embedding_model.encode(
     [formatted_query],
     batch_size=DENSE_CFG["settings"]["embedding"]["query_batch_size"],
     output_value="sentence_embedding",
     convert_to_numpy=True,
     precision=DENSE_CFG["settings"]["embedding"]["precision"],
     normalize_embeddings=DENSE_CFG["settings"]["retrieval"]["normalize_embeddings"]
    )[0]

    return embed_query

In [ ]:
from src.retrieval import exact_cosine_search

def dense_search(query, k=None, exclude_ids=frozenset()) -> list[str]:
    if not isinstance(query, str):
        raise TypeError("query must be a string")
    if k is None:
        k = DENSE_K
    if not DENSE_CFG["settings"]["retrieval"]["document_mask"] and exclude_ids:
        raise ValueError("Document mask disabled in config; cannot exclude IDs")
    query_embedding = embed_query(query)
    stored = collection.get(include=["embeddings"], limit=collection.count())
    return exact_cosine_search(
        doc_ids=stored["ids"],
        document_embeddings=stored["embeddings"],
        query_embedding=query_embedding,
        k=k,
        exclude_ids=exclude_ids,
    )


In [ ]:
test_query = questions_dev[0]["question"]

ranked_10 = dense_search(test_query, k=DENSE_K)

assert len(ranked_10) == DENSE_K
assert len(set(ranked_10)) == DENSE_K

masked_10 = dense_search(
    test_query,
    k=DENSE_K,
    exclude_ids={ranked_10[0]},
)

assert len(masked_10) == DENSE_K
assert ranked_10[0] not in masked_10
assert masked_10[:DENSE_K - 1] == ranked_10[1:DENSE_K]

print("Dense retrieval smoke test: PASSED")
print(f"Retrieved {DENSE_K} unique documents.")
print("Document masking: PASSED")

Question: Did the movies Ride A Crooked Trail and The Glassworker, originate from the same country?

Top-10 results:

 1. p02917 | Ride a Crooked Trail
 2. p02916 | Ride a Crooked Mile
 3. p03365 | The Glassworker
 4. p03320 | The Crooked Trail
 5. p03691 | Walk a Crooked Mile
 6. p02918 | Ride, Vaquero!
 7. p01334 | Glass Trap
 8. p03452 | The Rough and the Smooth
 9. p03412 | The Masked Rider (1919 film)
10. p03466 | The Sick Stockrider

Document-mask test: PASSED
Returned 10 results after excluding the original top-1.


In [ ]:
RUN_DIR = retrieval_path("dev").parent
RUN_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = retrieval_path("dev")
PRED_PATH.unlink(missing_ok=True)

with open(PRED_PATH, "w", encoding="utf-8") as f:
    for i, q in enumerate(questions_dev, start=1):
        retrieved = dense_search(q["question"], k=DENSE_K)
        if len(retrieved) != DENSE_K or len(set(retrieved)) != DENSE_K:
            raise RuntimeError(f"Invalid retrieval result for {q['qid']}: {retrieved}")
        row = {"qid": q["qid"], "answer": "", "retrieved": retrieved}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        if i % 25 == 0:
            print(f"Recomputed retrieval for {i}/300 questions")

fresh_retrieval = load_jsonl(PRED_PATH)
assert len(fresh_retrieval) == len(questions_dev) == 300
assert {row["qid"] for row in fresh_retrieval} == {q["qid"] for q in questions_dev}
print("Fresh Dense retrieval-only predictions:", len(fresh_retrieval))
print("Saved:", PRED_PATH)


Recomputed retrieval for 25/300 questions
Recomputed retrieval for 50/300 questions
Recomputed retrieval for 75/300 questions
Recomputed retrieval for 100/300 questions
Recomputed retrieval for 125/300 questions
Recomputed retrieval for 150/300 questions
Recomputed retrieval for 175/300 questions
Recomputed retrieval for 200/300 questions
Recomputed retrieval for 225/300 questions
Recomputed retrieval for 250/300 questions
Recomputed retrieval for 275/300 questions
Recomputed retrieval for 300/300 questions
Fresh Dense retrieval-only predictions: 300
Saved: /kaggle/working/runs/dev/dense_chroma_qwen3_4b/predictions.jsonl


In [ ]:
SCORE_STDOUT = run_score_cli(
    PRED_PATH,
    split="dev",
    stage_name="Dense retrieval-only / dev",
)


=== Official score.py: Dense retrieval-only / dev ===
valid: /kaggle/working/runs/dev/dense_chroma_qwen3_4b/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2          65.8   [ 62.9,  68.4]
  hitall@2          32.7   [ 27.0,  38.0]
  recall@5          75.8   [ 73.0,  78.5]
  hitall@5          50.0   [ 44.3,  55.3]
  recall@10         79.6   [ 76.9,  82.2]
  hitall@10         57.3   [ 51.7,  63.0]
  mrr               97.7   [ 96.3,  98.8]
  ndcg@10           80.3   [ 78.3,  82.3]
  em                 0.0   [  0.0,   0.0]
  f1                 0.0   [  0.0,   0.0]

type: bridge_comparison (n=66)
  recall@2          47.7   [ 45.8,  49.2]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5          54.2   [ 51.9,  56.8]
  hitall@5           1.5   [  0.0,   4.5]
  recall@10         54.9   [ 52.3,  57.6]
  hitall@10          1.5   [  0.0,   4.5]
  mrr               99.2   [ 97.7, 100.0]
  ndcg@10           66.4   [ 64.5,  68.4]
  em                 0.0   [  0.0,   0.0]
  f1    

In [ ]:
comparison_report = compare_retrieval_metrics_with_score(
    predictions_path=PRED_PATH,
    score_stdout=SCORE_STDOUT,
    labels_by_qid=label_map,
    stage_name='Dense retrieval-only / dev',
)


=== Manual comparison: Dense retrieval-only / dev (n=300) ===
metric           ours (%)   score.py (%)   delta (pp)             status
------------------------------------------------------------------------
recall@2          65.8333           65.8       0.0333   MATCH (rounding)
hitall@2          32.6667           32.7      -0.0333   MATCH (rounding)
recall@5          75.7500           75.8      -0.0500   MATCH (rounding)
hitall@5          50.0000           50.0       0.0000   MATCH (rounding)
recall@10         79.5833           79.6      -0.0167   MATCH (rounding)
hitall@10         57.3333           57.3       0.0333   MATCH (rounding)
mrr               97.6667           97.7      -0.0333   MATCH (rounding)
ndcg@10           80.3281           80.3       0.0281   MATCH (rounding)
Result: all eight retrieval metrics match score.py within output rounding.


# DenseRAG + LLM

In [ ]:
RETRIEVAL_PATH = retrieval_path("dev")

with open(RETRIEVAL_PATH, encoding="utf-8") as f:
    retrieval_rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

question_by_qid = {
    q["qid"]: q
    for q in questions_dev
}

assert len(retrieval_rows) == 300
assert all(row["qid"] in question_by_qid for row in retrieval_rows)
assert all(len(row["retrieved"]) == DENSE_K for row in retrieval_rows)

print("Questions:", len(retrieval_rows))
print("Retrieved documents per question:", DENSE_K)
print("First question:", question_by_qid[retrieval_rows[0]["qid"]]["question"])
print("Retrieved IDs:", retrieval_rows[0]["retrieved"])


Questions: 300
Retrieved documents per question: 10
First question: Did the movies Ride A Crooked Trail and The Glassworker, originate from the same country?
Retrieved IDs: ['p02917', 'p02916', 'p03365', 'p03320', 'p03691', 'p02918', 'p01334', 'p03452', 'p03412', 'p03466']


In [ ]:
if "embedding_model" in globals():
    del embedding_model

gc.collect()
torch.cuda.empty_cache()


In [ ]:
doc_by_id = {
    str(d["doc_id"]): d
    for d in corpus
}

SYSTEM_PROMPT = load_prompt(DENSE_CFG)
MAX_INPUT_TOKENS = DENSE_CFG["settings"]["generation"]["max_input_tokens"]

def build_rag_messages(question, doc_ids):

    passage_blocks = []

    for rank, doc_id in enumerate(doc_ids, start=1):
        if doc_id not in doc_by_id:
            raise KeyError(f"Unknown document ID: {doc_id}")

        doc = doc_by_id[doc_id]

        passage_blocks.append(
            f"[Passage {rank} | {doc_id}]\n"
            f"Title: {doc['title']}\n"
            f"Text: {doc['text']}"
        )

    passages_text = "\n\n".join(passage_blocks)

    user_prompt = (
        f"Passages:\n\n{passages_text}\n\n"
        f"Question: {question}\n\n"
        "Final answer:"
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]


def tokenize_rag_prompt(question, doc_ids):
    for n in range(min(len(doc_ids), DENSE_CFG["settings"]["generation"]["context_top_k"]), 0, -1):

        messages = build_rag_messages(
            question,
            doc_ids[:n]
        )

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=DENSE_CFG["settings"]["generation"]["enable_thinking"],
            return_dict=True,
            return_tensors="pt"
        )

        input_length = inputs["input_ids"].shape[1]

        if input_length <= MAX_INPUT_TOKENS:
            return inputs, n

    raise ValueError(
        "The prompt exceeds the input-token limit, "
        "even with only the top-ranked passage."
    )

def generate_dense_rag_answer(
    question,
    doc_ids,
    max_new_tokens=None
):

    if max_new_tokens is None:
        max_new_tokens = DENSE_CFG["settings"]["generation"]["max_new_tokens"]

    if "llm" not in globals() or "processor" not in globals():
        raise RuntimeError(
            "Load Qwen3.5-9B and its processor first."
        )

    inputs, n_context = tokenize_rag_prompt(
        question,
        doc_ids
    )

    inputs = {
        key: value.to(DENSE_CFG["settings"]["generation"]["device"])
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=DENSE_CFG["settings"]["generation"]["do_sample"],
            pad_token_id=processor.tokenizer.eos_token_id
        )

    generated_ids = outputs[0][input_length:]

    answer_text = processor.tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    for prefix in ("Final answer:", "Answer:"):
        if answer_text.lower().startswith(prefix.lower()):
            answer_text = answer_text[len(prefix):].strip()

    answer_text = answer_text.splitlines()[0].strip() if answer_text else ""

    return answer_text


In [ ]:
MODEL_ID = DENSE_CFG["models"]["llm"]["name"]

assert torch.cuda.is_available(), (
    "GPU is not enabled. Go to Kaggle Settings > Accelerator."
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=DENSE_CFG["models"]["llm"]["load_in_4bit"],
    bnb_4bit_quant_type=DENSE_CFG["models"]["llm"]["quantization"],
    bnb_4bit_compute_dtype=get_torch_dtype(torch, DENSE_CFG),
    bnb_4bit_use_double_quant=DENSE_CFG["models"]["llm"]["double_quantization"]
)

print("Loading processor...")

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

print("Loading Qwen3.5-9B...")

llm = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=getattr(torch, DENSE_CFG["models"]["llm"]["model_dtype"]),
    device_map=DENSE_CFG["models"]["llm"]["device_map"],
    attn_implementation=DENSE_CFG["models"]["llm"]["attention_implementation"],
    low_cpu_mem_usage=DENSE_CFG["models"]["llm"]["low_cpu_mem_usage"]
)

llm.eval()

print("Model loaded successfully.")
print("Model device:", llm.device)
print("Processor:", type(processor).__name__)


Loading processor...
Loading Qwen3.5-9B...


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Model loaded successfully.
Model device: cuda:0
Processor: Qwen3VLProcessor


In [ ]:
DATA_DIR = Path(DATA_DIR)
RETRIEVAL_PATH = retrieval_path("dev")
OUT_DIR = prediction_path(DENSE_CFG, "dev").parent
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "predictions.jsonl"

assert RETRIEVAL_PATH.is_file(), "Fresh Dense retrieval file not found. Run Dense retrieval stage first."
assert "llm" in globals(), "Load the Qwen LLM first."
assert "processor" in globals(), "Load the processor first."
assert "generate_dense_rag_answer" in globals(), "Define generate_dense_rag_answer first."

questions_dev = load_jsonl(DATA_DIR / "questions_dev.jsonl")
retrieval_rows = load_jsonl(RETRIEVAL_PATH)
assert len(questions_dev) == len(retrieval_rows) == 300
question_by_qid = {q["qid"]: q for q in questions_dev}
retrieval_by_qid = {row["qid"]: row for row in retrieval_rows}
assert len(question_by_qid) == len(retrieval_by_qid) == 300
assert set(retrieval_by_qid) == set(question_by_qid)
assert all(len(row["retrieved"]) == DENSE_K for row in retrieval_rows)

OUT_PATH.unlink(missing_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for i, q in enumerate(questions_dev, start=1):
        retrieved = retrieval_by_qid[q["qid"]]["retrieved"]
        answer_text = generate_dense_rag_answer(
            question=q["question"],
            doc_ids=retrieved,
            max_new_tokens=DENSE_CFG["settings"]["generation"]["max_new_tokens"],
        )
        row = {"qid": q["qid"], "answer": str(answer_text).strip(), "retrieved": retrieved}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        if i % 10 == 0:
            print(f"Fresh Dense RAG: {i}/300 | latest answer={answer_text[:100]!r}")

fresh_dense_rag = load_jsonl(OUT_PATH)
assert len(fresh_dense_rag) == 300
assert {row["qid"] for row in fresh_dense_rag} == set(question_by_qid)
print("Fresh Dense RAG / dev completed:", len(fresh_dense_rag))
print("Predictions:", OUT_PATH)


Fresh Dense RAG: 10/300 | latest answer='Georgetown, Kentucky'
Fresh Dense RAG: 20/300 | latest answer='George Venables-Vernon, 1st Baron Vernon'
Fresh Dense RAG: 30/300 | latest answer='Love And Hisses'
Fresh Dense RAG: 40/300 | latest answer='Jason Scott'
Fresh Dense RAG: 50/300 | latest answer='French'
Fresh Dense RAG: 60/300 | latest answer='No information provided.'
Fresh Dense RAG: 70/300 | latest answer='Brodick Castle'
Fresh Dense RAG: 80/300 | latest answer='Lorenzo Zazzeri'
Fresh Dense RAG: 90/300 | latest answer='Queen Munjeong'
Fresh Dense RAG: 100/300 | latest answer='Australian'
Fresh Dense RAG: 110/300 | latest answer='Prince Viliami Tungī Mailefihi'
Fresh Dense RAG: 120/300 | latest answer='Madrid'
Fresh Dense RAG: 130/300 | latest answer='Spain'
Fresh Dense RAG: 140/300 | latest answer='Yes'
Fresh Dense RAG: 150/300 | latest answer='September 11, 1765'
Fresh Dense RAG: 160/300 | latest answer='Milan'
Fresh Dense RAG: 170/300 | latest answer='France'
Fresh Dense RAG: 18

In [ ]:
SCORE_STDOUT = run_score_cli(
    OUT_PATH,
    split="dev",
    stage_name="Dense RAG / dev",
)


=== Official score.py: Dense RAG / dev ===
valid: /kaggle/working/runs/dev/dense_rag_qwen3_5_9b/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2          65.8   [ 62.9,  68.4]
  hitall@2          32.7   [ 27.0,  38.0]
  recall@5          75.8   [ 73.0,  78.5]
  hitall@5          50.0   [ 44.3,  55.3]
  recall@10         79.6   [ 76.9,  82.2]
  hitall@10         57.3   [ 51.7,  63.0]
  mrr               97.7   [ 96.3,  98.8]
  ndcg@10           80.3   [ 78.3,  82.3]
  em                52.7   [ 47.3,  58.3]
  f1                58.9   [ 54.0,  63.8]

type: bridge_comparison (n=66)
  recall@2          47.7   [ 45.8,  49.2]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5          54.2   [ 51.9,  56.8]
  hitall@5           1.5   [  0.0,   4.5]
  recall@10         54.9   [ 52.3,  57.6]
  hitall@10          1.5   [  0.0,   4.5]
  mrr               99.2   [ 97.7, 100.0]
  ndcg@10           66.4   [ 64.5,  68.4]
  em                68.2   [ 57.6,  78.8]
  f1                

In [ ]:
comparison_report = compare_retrieval_metrics_with_score(
    predictions_path=OUT_PATH,
    score_stdout=SCORE_STDOUT,
    labels_by_qid=label_map,
    stage_name='Dense RAG / dev',
)



=== Manual comparison: Dense RAG / dev (n=300) ===
metric           ours (%)   score.py (%)   delta (pp)             status
------------------------------------------------------------------------
recall@2          65.8333           65.8       0.0333   MATCH (rounding)
hitall@2          32.6667           32.7      -0.0333   MATCH (rounding)
recall@5          75.7500           75.8      -0.0500   MATCH (rounding)
hitall@5          50.0000           50.0       0.0000   MATCH (rounding)
recall@10         79.5833           79.6      -0.0167   MATCH (rounding)
hitall@10         57.3333           57.3       0.0333   MATCH (rounding)
mrr               97.6667           97.7      -0.0333   MATCH (rounding)
ndcg@10           80.3281           80.3       0.0281   MATCH (rounding)
Result: all eight retrieval metrics match score.py within output rounding.


In [43]:
embedding_name = DENSE_CFG["models"]["embedder"]

embedding_model = SentenceTransformer(
    embedding_name,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print("Embedding model loaded:", embedding_name)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Embedding model loaded: Qwen/Qwen3-Embedding-4B
Device: cuda


In [ ]:
PACK_DIR = DATA_DIR
QUESTIONS_PATH = PACK_DIR / "questions_hidden.jsonl"
RETRIEVAL_PATH = retrieval_path("hidden")
RETRIEVAL_DIR = RETRIEVAL_PATH.parent
RETRIEVAL_DIR.mkdir(parents=True, exist_ok=True)

assert "embedding_model" in globals(), "Embedding model is not loaded."
assert "dense_search" in globals(), "dense_search is not defined."
assert "doc_by_id" in globals(), "doc_by_id is not defined."

questions_hidden = load_jsonl(QUESTIONS_PATH)
assert len(questions_hidden) == 300, f"Expected 300 questions, got {len(questions_hidden)}"
RETRIEVAL_PATH.unlink(missing_ok=True)

with open(RETRIEVAL_PATH, "w", encoding="utf-8") as f:
    for i, q in enumerate(tqdm(questions_hidden, desc="Fresh Dense retrieval on Hidden"), start=1):
        doc_ids = dense_search(q["question"], k=DENSE_K)
        if len(doc_ids) != DENSE_K or len(set(doc_ids)) != DENSE_K:
            raise RuntimeError(f"{q['qid']}: invalid top-{DENSE_K} retrieval")
        if not all(doc_id in doc_by_id for doc_id in doc_ids):
            raise RuntimeError(f"{q['qid']}: unknown document ID")
        f.write(json.dumps({"qid": q["qid"], "retrieved": doc_ids}, ensure_ascii=False) + "\n")
        f.flush()

fresh_hidden_retrieval = load_jsonl(RETRIEVAL_PATH)
assert len(fresh_hidden_retrieval) == 300
assert {row["qid"] for row in fresh_hidden_retrieval} == {q["qid"] for q in questions_hidden}
print("Fresh Hidden retrieval completed:", len(fresh_hidden_retrieval))
print("Saved to:", RETRIEVAL_PATH)


Fresh Dense retrieval on Hidden:   0%|          | 0/300 [00:00<?, ?it/s]

Fresh Hidden retrieval completed: 300
Saved to: /kaggle/working/runs/hidden/dense_chroma_qwen3_4b/predictions.jsonl


In [ ]:
del embedding_model

gc.collect()
torch.cuda.empty_cache()

In [ ]:
MODEL_ID = DENSE_CFG["models"]["llm"]["name"]

if "processor" not in globals():
    processor = AutoProcessor.from_pretrained(MODEL_ID)

quant_config = BitsAndBytesConfig(
    load_in_4bit=DENSE_CFG["models"]["llm"]["load_in_4bit"],
    bnb_4bit_quant_type=DENSE_CFG["models"]["llm"]["quantization"],
    bnb_4bit_compute_dtype=get_torch_dtype(torch, DENSE_CFG),
    bnb_4bit_use_double_quant=DENSE_CFG["models"]["llm"]["double_quantization"]
)

llm = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=getattr(torch, DENSE_CFG["models"]["llm"]["model_dtype"]),
    device_map=DENSE_CFG["models"]["llm"]["device_map"],
    attn_implementation=DENSE_CFG["models"]["llm"]["attention_implementation"],
    low_cpu_mem_usage=DENSE_CFG["models"]["llm"]["low_cpu_mem_usage"]
)

llm.eval()

print("LLM loaded successfully.")
print("Device:", llm.device)

Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

LLM loaded successfully.
Device: cuda:0


In [ ]:
OUT_DIR = prediction_path(DENSE_CFG, "hidden").parent
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "predictions.jsonl"

retrieval_rows = load_jsonl(RETRIEVAL_PATH)
retrieval_by_qid = {row["qid"]: row["retrieved"] for row in retrieval_rows}
assert len(retrieval_by_qid) == 300
assert set(retrieval_by_qid) == {q["qid"] for q in questions_hidden}

OUT_PATH.unlink(missing_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for i, q in enumerate(tqdm(questions_hidden, desc="Fresh Dense RAG answers on Hidden"), start=1):
        answer = generate_dense_rag_answer(
            question=q["question"],
            doc_ids=retrieval_by_qid[q["qid"]],
            max_new_tokens=DENSE_CFG["settings"]["generation"]["max_new_tokens"],
        )
        if isinstance(answer, dict):
            answer = answer.get("answer", "")
        row = {
            "qid": q["qid"],
            "answer": str(answer).strip(),
            "retrieved": retrieval_by_qid[q["qid"]],
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()

fresh_hidden_answers = load_jsonl(OUT_PATH)
assert len(fresh_hidden_answers) == 300
assert {row["qid"] for row in fresh_hidden_answers} == {q["qid"] for q in questions_hidden}
print("Fresh Hidden Dense RAG completed:", len(fresh_hidden_answers))
print("Predictions:", OUT_PATH)


Fresh Dense RAG answers on Hidden:   0%|          | 0/300 [00:00<?, ?it/s]

Fresh Hidden Dense RAG completed: 300
Predictions: /kaggle/working/runs/hidden/dense_rag_qwen3_5_9b/predictions.jsonl


In [ ]:
SCORE_STDOUT = run_score_cli(
    OUT_PATH,
    split="hidden",
    validate_only=True,
    stage_name="Dense RAG / hidden validation",
)


=== Official score.py: Dense RAG / hidden validation ===
valid: /kaggle/working/runs/hidden/dense_rag_qwen3_5_9b/predictions.jsonl (300 predictions)


### Dense RAG / hidden: metric comparison intentionally skipped

`score.py --validate-only` checks the prediction schema and IDs. We do not calculate or compare retrieval metrics for Hidden because its gold labels are not available in this notebook.


In [ ]:
ARTIFACT_DIR = Path("/kaggle/working/analysis_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

dense_dev = prediction_path(DENSE_CFG, "dev")
closed_dev = prediction_path(CLOSED_CFG, "dev")
oracle_dev = prediction_path(ORACLE_CFG, "dev")

for path in [dense_dev, closed_dev, oracle_dev]:
    assert path.is_file(), f"Missing prediction file: {path}"

dense_vs_closed = run_score_cli(
    predictions_path=dense_dev,
    split="dev",
    compare_path=closed_dev,
    stage_name="Dense RAG vs Closed-book",
)

(ARTIFACT_DIR / "dense_rag_minus_closed_book.txt").write_text(
    dense_vs_closed,
    encoding="utf-8",
)

dense_vs_oracle = run_score_cli(
    predictions_path=dense_dev,
    split="dev",
    compare_path=oracle_dev,
    stage_name="Dense RAG vs Oracle-context",
)

(ARTIFACT_DIR / "dense_rag_minus_oracle_context.txt").write_text(
    dense_vs_oracle,
    encoding="utf-8",
)

print("\nPaired comparison outputs saved to:", ARTIFACT_DIR)



=== Official score.py: Dense RAG vs Closed-book ===
valid: /kaggle/working/runs/dev/dense_rag_qwen3_5_9b/predictions.jsonl (300 predictions)
valid: /kaggle/working/runs/dev/closed_book/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2          65.8   [ 62.9,  68.4]
  hitall@2          32.7   [ 27.0,  38.0]
  recall@5          75.8   [ 73.0,  78.5]
  hitall@5          50.0   [ 44.3,  55.3]
  recall@10         79.6   [ 76.9,  82.2]
  hitall@10         57.3   [ 51.7,  63.0]
  mrr               97.7   [ 96.3,  98.8]
  ndcg@10           80.3   [ 78.3,  82.3]
  em                52.7   [ 47.3,  58.3]
  f1                58.9   [ 54.0,  63.8]

type: bridge_comparison (n=66)
  recall@2          47.7   [ 45.8,  49.2]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5          54.2   [ 51.9,  56.8]
  hitall@5           1.5   [  0.0,   4.5]
  recall@10         54.9   [ 52.3,  57.6]
  hitall@10          1.5   [  0.0,   4.5]
  mrr               99.2   [ 97.7, 100.0]
  ndcg@10      

## Dev error analysis: five cases per system and ten cross-system disagreements

This section analyzes only the labeled **Dev** split. It reads the already-saved predictions for Closed-book, Oracle-context, and Dense RAG; it does **not** run any model. All EM/F1 values come from the course `score.py` implementation. Failure labels are evidence-based heuristics for manual review, not proof of root cause.

It creates CSV, JSON, and Markdown files with:

- five incorrect/diagnostic cases for each system (15 cases total), including partial answers, likely retrieval-coverage gaps for Dense RAG, and likely generation/reasoning failures;
- ten distinct questions where at least one system is correct and another is incorrect, with each system's answer, EM/F1, retrieved IDs/titles, and missing gold passage IDs;
- a compact summary of the three systems' Dev metrics.

The ten disagreement cases are selected from actual prediction differences; no cases are fabricated. Hidden data is not used for labeled error analysis.

In [ ]:
import csv
import importlib.util
from collections import Counter, defaultdict

ANALYSIS_DIR = Path("/kaggle/working/analysis_artifacts")
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

_score_spec = importlib.util.spec_from_file_location("course_score_for_error_analysis", SCORE_PATH)
course_score = importlib.util.module_from_spec(_score_spec)
_score_spec.loader.exec_module(course_score)

questions_dev_for_analysis = load_jsonl(DATA_DIR / "questions_dev.jsonl")
labels_dev_for_analysis = load_jsonl(DATA_DIR / "labels_dev.jsonl")
corpus_for_analysis = load_jsonl(DATA_DIR / "corpus.jsonl")
question_by_qid_analysis = {row["qid"]: row for row in questions_dev_for_analysis}
label_by_qid_analysis = {row["qid"]: row for row in labels_dev_for_analysis}
doc_by_id_analysis = {str(row["doc_id"]): row for row in corpus_for_analysis}
expected_qids_analysis = set(question_by_qid_analysis)

SYSTEM_PATHS = {
    "closed_book": prediction_path(CLOSED_CFG, "dev"),
    "oracle_context": prediction_path(ORACLE_CFG, "dev"),
    "dense_rag": prediction_path(DENSE_CFG, "dev"),
}
SYSTEM_LABELS = {
    "closed_book": "Closed-book",
    "oracle_context": "Oracle-context",
    "dense_rag": "Dense RAG",
}

predictions_by_system = {}
metrics_by_system = {}
for system_key, prediction_path in SYSTEM_PATHS.items():
    if not prediction_path.is_file():
        raise FileNotFoundError(
            f"Missing saved Dev predictions for {SYSTEM_LABELS[system_key]}: {prediction_path}. "
            "Run the corresponding prediction cell first; this analysis does not regenerate predictions."
        )
    rows = load_jsonl(prediction_path)
    by_qid = {row["qid"]: row for row in rows}
    if len(rows) != 300 or set(by_qid) != expected_qids_analysis:
        raise AssertionError(
            f"{SYSTEM_LABELS[system_key]} must contain exactly one prediction for each of the 300 Dev qids."
        )
    predictions_by_system[system_key] = by_qid
    metrics_by_system[system_key] = {
        qid: course_score.per_question(by_qid[qid], label_by_qid_analysis[qid])
        for qid in sorted(expected_qids_analysis)
    }


def _titles_for_ids(doc_ids):
    result = []
    for doc_id in doc_ids:
        key = str(doc_id)
        doc = doc_by_id_analysis.get(key)
        result.append({"doc_id": key, "title": doc.get("title", "[title unavailable]") if doc else "[document not found]"})
    return result


def _diagnostic_record(system_key, qid):
    pred = predictions_by_system[system_key][qid]
    label = label_by_qid_analysis[qid]
    question = question_by_qid_analysis[qid]
    metrics = metrics_by_system[system_key][qid]
    answer = "" if pred.get("abstained") else str(pred.get("answer", ""))
    retrieved = [str(doc_id) for doc_id in pred.get("retrieved", [])]
    gold_ids = sorted(str(doc_id) for doc_id in label["gold_doc_ids"])
    retrieved_top10 = retrieved[:10]
    retrieved_set = set(retrieved_top10)
    missing_gold = sorted(set(gold_ids) - retrieved_set)
    em = float(metrics["em"])
    f1_value = float(metrics["f1"])

    if system_key == "closed_book":
        retrieval_recall = None
        hitall10 = None
        missing_for_report = []  # retrieval is intentionally disabled in Closed-book
        retrieval_assessment = "N/A: Closed-book intentionally does not retrieve passages."
        if not answer.strip():
            category = "Empty/abstained answer"
            note = "The system returned no answer; inspect generation and answer extraction."
        elif f1_value > 0:
            category = "Partially correct answer (EM=0, F1>0)"
            note = "Some answer tokens overlap with the reference, but the answer is not an exact match under the course scorer."
        else:
            category = "Closed-book knowledge/generation mismatch"
            note = "The answer has no token overlap with the reference under the scorer; without retrieved context, this is consistent with a knowledge or generation failure."
    else:
        retrieval_recall = len(set(gold_ids) & retrieved_set) / len(gold_ids) if gold_ids else 0.0
        hitall10 = float(set(gold_ids).issubset(retrieved_set))
        missing_for_report = missing_gold
        retrieval_assessment = "All gold passages in top 10." if not missing_gold else "Gold passage coverage incomplete in top 10."
        if system_key == "dense_rag" and missing_gold:
            category = "Partial answer + retrieval coverage gap" if f1_value > 0 else "Retrieval coverage gap (missing gold passage(s))"
            note = (
                "At least one gold passage is absent from the top 10, so retrieval coverage is a plausible contributor; "
                "this alone does not prove it is the only cause. Inspect the retrieved titles and context truncation."
            )
        elif not answer.strip():
            category = "Empty/abstained answer"
            note = "The system returned no answer despite receiving context; inspect generation and answer extraction."
        elif f1_value > 0:
            category = "Partially correct answer (EM=0, F1>0)"
            note = "The answer overlaps with the reference but is not exact. Inspect evidence integration, reasoning, and answer normalization."
        elif system_key == "oracle_context":
            category = "LLM reasoning/output failure despite oracle evidence"
            note = "The Oracle system receives the gold passages; the mismatch points toward evidence use, reasoning, answer extraction, or answer normalization rather than retrieval."
        else:
            category = "LLM generation/reasoning failure despite complete gold evidence" if not missing_gold else "Answer-generation failure with incomplete retrieval"
            note = (
                "All gold passage IDs are present in the top 10, but the answer has no scorer overlap. "
                "Inspect context packing/truncation and generation before assigning a definitive root cause."
                if not missing_gold else
                "The answer has no scorer overlap and some gold evidence is missing; retrieval and generation may both contribute."
            )

    return {
        "system": SYSTEM_LABELS[system_key],
        "system_key": system_key,
        "qid": qid,
        "question_type": question.get("type", label.get("type", "unknown")),
        "question": question["question"],
        "gold_answer": label["answer"],
        "gold_aliases": label.get("aliases", []),
        "system_answer": answer,
        "em": em,
        "f1": f1_value,
        "gold_doc_ids": gold_ids,
        "retrieved_doc_ids_top10": retrieved_top10,
        "retrieved_passages_top10": _titles_for_ids(retrieved_top10),
        "missing_gold_doc_ids_top10": missing_for_report,
        "retrieval_recall_at_10": retrieval_recall,
        "hitall_at_10": hitall10,
        "retrieval_assessment": retrieval_assessment,
        "failure_category_heuristic": category,
        "diagnostic_note": note,
        "diagnosis_caveat": "Heuristic for human review; not a proven causal diagnosis.",
    }

all_error_records = {system_key: [] for system_key in SYSTEM_PATHS}
for system_key in SYSTEM_PATHS:
    for qid in sorted(expected_qids_analysis):
        record = _diagnostic_record(system_key, qid)
        if record["em"] < 1.0:
            all_error_records[system_key].append(record)

def _error_bucket(system_key, record):
    if system_key == "dense_rag" and record["missing_gold_doc_ids_top10"]:
        return 0  
    if record["f1"] > 0:
        return 1  
    if not record["system_answer"].strip():
        return 2  
    return 3  


def _select_five_errors(system_key, records):
    if len(records) < 5:
        raise RuntimeError(
            f"Only {len(records)} incorrect Dev answers found for {SYSTEM_LABELS[system_key]}; cannot provide five error cases."
        )
    selected, used_qids, used_types = [], set(), set()
    buckets = defaultdict(list)
    for record in records:
        buckets[_error_bucket(system_key, record)].append(record)
    for bucket in sorted(buckets):
        candidates = sorted(
            buckets[bucket],
            key=lambda r: (r["question_type"] in used_types, -r["f1"], r["qid"]),
        )
        candidate = next((r for r in candidates if r["qid"] not in used_qids and r["question_type"] not in used_types), None)
        if candidate is None:
            candidate = next((r for r in candidates if r["qid"] not in used_qids), None)
        if candidate is not None and len(selected) < 5:
            selected.append(candidate)
            used_qids.add(candidate["qid"])
            used_types.add(candidate["question_type"])
    remaining = sorted(
        [r for r in records if r["qid"] not in used_qids],
        key=lambda r: (r["question_type"] in used_types, _error_bucket(system_key, r), -r["f1"], r["qid"]),
    )
    while len(selected) < 5 and remaining:
        candidate = next((r for r in remaining if r["question_type"] not in used_types), remaining[0])
        selected.append(candidate)
        used_qids.add(candidate["qid"])
        used_types.add(candidate["question_type"])
        remaining = [r for r in remaining if r["qid"] not in used_qids]
    for i, record in enumerate(selected, start=1):
        record["case_number_within_system"] = i
    return selected


error_cases = []
for system_key in SYSTEM_PATHS:
    selected = _select_five_errors(system_key, all_error_records[system_key])
    error_cases.extend(selected)

SYSTEM_ORDER = ["closed_book", "oracle_context", "dense_rag"]
SYSTEM_SHORT = {"closed_book": "closed_book", "oracle_context": "oracle_context", "dense_rag": "dense_rag"}
correctness = {
    qid: {system_key: bool(metrics_by_system[system_key][qid]["em"] == 1.0) for system_key in SYSTEM_ORDER}
    for qid in sorted(expected_qids_analysis)
}

def _signature(qid):
    return tuple(int(correctness[qid][system_key]) for system_key in SYSTEM_ORDER)

def _f1_spread(qid):
    vals = [float(metrics_by_system[s][qid]["f1"]) for s in SYSTEM_ORDER]
    return max(vals) - min(vals)

candidate_qids = [qid for qid in sorted(expected_qids_analysis) if len(set(correctness[qid].values())) > 1]
if len(candidate_qids) < 10:
    raise RuntimeError(f"Only {len(candidate_qids)} cross-system correctness disagreements found; cannot select ten unique cases.")

conditions = [
    ("dense_rag", True, "closed_book", False),
    ("dense_rag", False, "closed_book", True),
    ("oracle_context", True, "dense_rag", False),
    ("oracle_context", False, "dense_rag", True),
    ("closed_book", True, "oracle_context", False),
    ("closed_book", False, "oracle_context", True),
]
selected_disagreement_qids = []
used_disagreement_types = set()
used_signatures = set()
for left_sys, left_correct, right_sys, right_correct in conditions:
    candidates = [
        qid for qid in candidate_qids
        if qid not in selected_disagreement_qids
        and correctness[qid][left_sys] is left_correct
        and correctness[qid][right_sys] is right_correct
    ]
    candidates.sort(key=lambda qid: (
        question_by_qid_analysis[qid].get("type", "unknown") in used_disagreement_types,
        _signature(qid) in used_signatures,
        -_f1_spread(qid),
        qid,
    ))
    if candidates and len(selected_disagreement_qids) < 10:
        chosen = candidates[0]
        selected_disagreement_qids.append(chosen)
        used_disagreement_types.add(question_by_qid_analysis[chosen].get("type", "unknown"))
        used_signatures.add(_signature(chosen))

remaining_qids = [qid for qid in candidate_qids if qid not in selected_disagreement_qids]
while len(selected_disagreement_qids) < 10:
    remaining_qids.sort(key=lambda qid: (
        question_by_qid_analysis[qid].get("type", "unknown") in used_disagreement_types,
        _signature(qid) in used_signatures,
        -_f1_spread(qid),
        qid,
    ))
    chosen = remaining_qids.pop(0)
    selected_disagreement_qids.append(chosen)
    used_disagreement_types.add(question_by_qid_analysis[chosen].get("type", "unknown"))
    used_signatures.add(_signature(chosen))


def _system_case_fields(system_key, qid):
    pred = predictions_by_system[system_key][qid]
    label = label_by_qid_analysis[qid]
    metrics = metrics_by_system[system_key][qid]
    retrieved = [str(x) for x in pred.get("retrieved", [])[:10]]
    gold = set(str(x) for x in label["gold_doc_ids"])
    missing = sorted(gold - set(retrieved)) if system_key != "closed_book" else []
    return {
        "answer": "" if pred.get("abstained") else str(pred.get("answer", "")),
        "em": float(metrics["em"]),
        "f1": float(metrics["f1"]),
        "retrieved_doc_ids_top10": retrieved,
        "retrieved_passages_top10": _titles_for_ids(retrieved),
        "missing_gold_doc_ids_top10": missing,
    }


disagreement_cases = []
for i, qid in enumerate(selected_disagreement_qids, start=1):
    question = question_by_qid_analysis[qid]
    label = label_by_qid_analysis[qid]
    systems = {system_key: _system_case_fields(system_key, qid) for system_key in SYSTEM_ORDER}
    correct_systems = [SYSTEM_LABELS[s] for s in SYSTEM_ORDER if systems[s]["em"] == 1.0]
    incorrect_systems = [SYSTEM_LABELS[s] for s in SYSTEM_ORDER if systems[s]["em"] == 0.0]
    disagreement_cases.append({
        "case_number": i,
        "qid": qid,
        "question_type": question.get("type", label.get("type", "unknown")),
        "question": question["question"],
        "gold_answer": label["answer"],
        "gold_aliases": label.get("aliases", []),
        "gold_doc_ids": sorted(str(x) for x in label["gold_doc_ids"]),
        "correct_systems_by_em": correct_systems,
        "incorrect_systems_by_em": incorrect_systems,
        "correctness_signature_closed_oracle_dense": list(_signature(qid)),
        "systems": systems,
    })

summary_rows = []
for system_key in SYSTEM_ORDER:
    per_q = [metrics_by_system[system_key][qid] for qid in sorted(expected_qids_analysis)]
    row = {"system": SYSTEM_LABELS[system_key], "n": len(per_q)}
    for metric_name in ("em", "f1", "recall@2", "hitall@2", "recall@5", "hitall@5", "recall@10", "hitall@10", "mrr", "ndcg@10"):
        row[metric_name] = sum(float(m[metric_name]) for m in per_q) / len(per_q)
    summary_rows.append(row)


def _csv_safe_row(row):
    return {
        key: (
            json.dumps(value, ensure_ascii=False)
            if isinstance(value, (list, dict))
            else value
        )
        for key, value in row.items()
    }


def _write_csv(path, rows):
    if not rows:
        raise ValueError(f"No rows to write: {path}")

    fieldnames = list(rows[0].keys())

    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
            extrasaction="ignore"
        )
        writer.writeheader()
        writer.writerows(_csv_safe_row(row) for row in rows)


_write_csv(
    ANALYSIS_DIR / "error_analysis_15_cases.csv",
    error_cases
)

_write_csv(
    ANALYSIS_DIR / "cross_system_disagreements_10.csv",
    [
        {
            "case_number": row["case_number"],
            "qid": row["qid"],
            "question_type": row["question_type"],
            "question": row["question"],
            "gold_answer": row["gold_answer"],
            "gold_aliases": row["gold_aliases"],
            "gold_doc_ids": row["gold_doc_ids"],
            "correct_systems_by_em": row["correct_systems_by_em"],
            "incorrect_systems_by_em": row["incorrect_systems_by_em"],
            **{
                f"{system_key}_{field}": row["systems"][system_key][field]
                for system_key in SYSTEM_ORDER
                for field in (
                    "answer",
                    "em",
                    "f1",
                    "retrieved_doc_ids_top10",
                    "retrieved_passages_top10",
                    "missing_gold_doc_ids_top10",
                )
            },
        }
        for row in disagreement_cases
    ]
)

_write_csv(
    ANALYSIS_DIR / "dev_system_summary.csv",
    summary_rows
)

print("CSV files saved to:", ANALYSIS_DIR)
print("Error cases:", len(error_cases))
print("Cross-system disagreements:", len(disagreement_cases))
print("Disagreement question types:",
      dict(Counter(row["question_type"] for row in disagreement_cases)))
print("Failure categories:",
      dict(Counter(row["failure_category_heuristic"] for row in error_cases)))

for row in summary_rows:
    print(
        f"{row['system']:<15} "
        f"EM={100 * row['em']:.1f}% "
        f"F1={100 * row['f1']:.1f}% "
        f"Recall@10={100 * row['recall@10']:.1f}% "
        f"HitAll@10={100 * row['hitall@10']:.1f}%"
    )

Created error-analysis artifacts in: /kaggle/working/analysis_artifacts
Error cases: 15 (5 per system)
Cross-system disagreements: 10 (10 unique qids)
Disagreement question types: {'bridge_comparison': 2, 'comparison': 4, 'compositional': 2, 'inference': 2}
Failure-category counts: {'Partially correct answer (EM=0, F1>0)': 9, 'Closed-book knowledge/generation mismatch': 1, 'LLM reasoning/output failure despite oracle evidence': 1, 'Partial answer + retrieval coverage gap': 1, 'LLM generation/reasoning failure despite complete gold evidence': 1, 'Retrieval coverage gap (missing gold passage(s))': 2}
Summary:
  Closed-book     EM=24.7% F1=29.6% Recall@10=0.0% HitAll@10=0.0%
  Oracle-context  EM=70.3% F1=77.0% Recall@10=100.0% HitAll@10=100.0%
  Dense RAG       EM=52.7% F1=58.9% Recall@10=79.6% HitAll@10=57.3%


In [ ]:
import zipfile
from datetime import datetime, timezone
from IPython.display import FileLink, display

WORK_ROOT = Path("/kaggle/working")
BUNDLE_PATH = WORK_ROOT / "phase1_analysis_bundle.zip"
MANIFEST_PATH = ANALYSIS_DIR / "bundle_manifest.txt"

known_prediction_paths = [
    prediction_path(CLOSED_CFG, "dev"),
    prediction_path(CLOSED_CFG, "hidden"),
    prediction_path(ORACLE_CFG, "dev"),
    prediction_path(DENSE_CFG, "dev"),
    prediction_path(DENSE_CFG, "hidden"),
    retrieval_path("dev"),
    retrieval_path("hidden"),
    WORK_ROOT / "llm_experiments" / "oracle_50" / "qwen3_5_9b" / "predictions.jsonl",
    WORK_ROOT / "llm_experiments" / "oracle_50" / "qwen3_5_9b" / "timings.jsonl",
    WORK_ROOT / "llm_experiments" / "oracle_50" / "sample_qids.json",
]
files_to_add = {}
def _add_file(path, archive_name=None):
    path = Path(path)
    if path.is_file():
        rel = archive_name or str(path.relative_to(WORK_ROOT)) if path.is_relative_to(WORK_ROOT) else (archive_name or path.name)
        files_to_add[str(rel)] = path

for prediction_path in known_prediction_paths:
    _add_file(prediction_path)
    if prediction_path.name == "predictions.jsonl":
        _add_file(prediction_path.with_name("run_config.yaml"))
        _add_file(prediction_path.with_name("timings.jsonl"))

for run_root in [WORK_ROOT / "runs", WORK_ROOT / "llm_experiments"]:
    if run_root.is_dir():
        for path in run_root.rglob("*"):
            if path.is_file() and path.name in {"predictions.jsonl", "run_config.yaml", "timings.jsonl", "sample_qids.json"}:
                _add_file(path)

for path in [
    PROJECT_ROOT / "score.py",
    PROJECT_ROOT / "predictions_format.md",
    PROJECT_ROOT / "src" / "config.py",
    PROJECT_ROOT / "src" / "metrics.py",
    PROJECT_ROOT / "src" / "retrieval.py",
]:
    _add_file(path, f"project_source/{path.name}")
for directory_name in ("configs", "prompts"):
    directory = PROJECT_ROOT / directory_name
    if directory.is_dir():
        for path in directory.rglob("*"):
            if path.is_file() and path.suffix.lower() in {".yaml", ".yml", ".txt", ".md"}:
                _add_file(path, str(Path("project_source") / directory_name / path.relative_to(directory)))

for path in ANALYSIS_DIR.rglob("*"):
    if path.is_file() and path != BUNDLE_PATH:
        _add_file(path, str(Path("analysis_artifacts") / path.relative_to(ANALYSIS_DIR)))
for path in WORK_ROOT.rglob("report.pdf"):
    if path.is_file():
        _add_file(path, str(Path("reports") / path.relative_to(WORK_ROOT)))

manifest_lines = [
    "Phase 1 analysis bundle",
    f"Created UTC: {datetime.now(timezone.utc).isoformat()}",
    "Includes saved prediction outputs and run configs when available; no model weights or vector indexes.",
    "Error analysis uses Dev labels only. Hidden outputs are included only as prediction/validation artifacts.",
    "",
    "Files included:",
]
manifest_lines.extend(f"- {name}" for name in sorted(files_to_add))
MANIFEST_PATH.write_text("\n".join(manifest_lines) + "\n", encoding="utf-8")
files_to_add[str(Path("analysis_artifacts") / MANIFEST_PATH.name)] = MANIFEST_PATH

if not (ANALYSIS_DIR / "dev_error_analysis_report.md").is_file():
    raise RuntimeError("Error-analysis report is missing. Run the Dev analysis cell before packaging.")
if len(files_to_add) == 0:
    raise RuntimeError("No files were found to package.")

BUNDLE_PATH.unlink(missing_ok=True)
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for archive_name, path in sorted(files_to_add.items()):
        archive.write(path, arcname=archive_name)

print("ZIP created:", BUNDLE_PATH)
print("Files included:", len(files_to_add))
print("ZIP size (MB):", round(BUNDLE_PATH.stat().st_size / (1024**2), 2))
display(FileLink(str(BUNDLE_PATH)))

ZIP created: /kaggle/working/phase1_analysis_bundle.zip
Files included: 32
ZIP size (MB): 0.09


/kaggle/working/phase1_analysis_bundle.zip